# Israeli Law LLM - Continued Pretraining

Fine-tune **DictaLM 2.0** (7B, Mistral-based Hebrew LLM) on ~140K Israeli legal documents
using **Unsloth + QLoRA** for memory-efficient training on Colab Pro A100.

**Setup:** Runtime > Change runtime type > **A100 GPU**

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

In [ ]:
from huggingface_hub import login

# Replace with your HuggingFace token (needs write access)
# Get one at: https://huggingface.co/settings/tokens
login(token="YOUR_HF_TOKEN_HERE")

## Configuration

In [ ]:
# === Configuration (optimized for A100 40GB) ===
MODEL_NAME = "dicta-il/dictalm2.0"          # Base Hebrew LLM (Mistral-based, 7B)
DATASET_NAME = "mufeedh28/israeli-law-pretrain"  # Private dataset on HF
OUTPUT_NAME = "mufeedh28/dictalm2-israeli-law"   # Where to push the trained model

MAX_SEQ_LENGTH = 2048   # Context window for training
LORA_RANK = 64          # LoRA rank
LORA_ALPHA = 16         # LoRA scaling factor

BATCH_SIZE = 4          # Per-device batch size (A100 40GB)
GRAD_ACCUM = 4          # Gradient accumulation (effective batch = 16)
LEARNING_RATE = 2e-4    # Peak learning rate
NUM_EPOCHS = 1          # 1 epoch for continued pretraining to avoid overfitting
WARMUP_STEPS = 100      # LR warmup steps
SAVE_STEPS = 500        # Save checkpoint every N steps
EVAL_STEPS = 500        # Evaluate every N steps
LOGGING_STEPS = 10      # Log metrics every N steps

## Load Model (4-bit Quantized)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect (bf16 on A100)
    load_in_4bit=True,    # QLoRA 4-bit quantization
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

## Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,        # Unsloth optimized - use 0
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    DATASET_NAME,
    data_files={"train": "pretrain.jsonl", "test": "eval.jsonl"},
    token=True,
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Train: {len(train_dataset):,} documents")
print(f"Eval:  {len(eval_dataset):,} documents")
print(f"\nSample (first 200 chars):")
print(train_dataset[0]["text"][:200])

## Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=LOGGING_STEPS,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        report_to="none",
        push_to_hub=True,
        hub_model_id="mufeedh28/dictalm2-israeli-law",
        hub_strategy="every_save",
        hub_private_repo=True,
    ),
)

print(f"Packing enabled - training will be efficient")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Model will auto-push to HuggingFace every {SAVE_STEPS} steps")

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Memory reserved: {start_gpu_memory} GB / {max_memory} GB")
print(f"\nStarting training...")

trainer_stats = trainer.train()

# Print final stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n{'='*50}")
print(f"Training complete!")
print(f"Peak GPU memory: {used_memory} GB / {max_memory} GB ({round(used_memory/max_memory*100, 1)}%)")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Total steps: {trainer_stats.global_step}")

## Save & Push to HuggingFace

In [ ]:
# Save LoRA adapters locally
model.save_pretrained("dictalm2-israeli-law-lora")
tokenizer.save_pretrained("dictalm2-israeli-law-lora")
print("Saved LoRA adapters locally")

# Push LoRA adapters to HuggingFace
model.push_to_hub(OUTPUT_NAME, token=True)
tokenizer.push_to_hub(OUTPUT_NAME, token=True)
print(f"Pushed LoRA adapters to: {OUTPUT_NAME}")

In [ ]:
# Save merged full model (16-bit) for standalone use
# This creates a complete model that doesn't need the base model
model.save_pretrained_merged(
    "dictalm2-israeli-law-merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Saved merged 16-bit model locally")

# Push merged model to HuggingFace (public - this is the open-source release)
model.push_to_hub_merged(
    f"{OUTPUT_NAME}-merged",
    tokenizer,
    save_method="merged_16bit",
    token=True,
)
print(f"Pushed merged model to: {OUTPUT_NAME}-merged")

## Test Generation

In [ ]:
# Quick test - generate some Hebrew legal text
FastLanguageModel.for_inference(model)

prompts = [
    "חוק יסוד: כבוד האדם וחירותו קובע כי",
    "בית המשפט העליון פסק כי",
    "על פי חוק השכירות והשאילה",
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.15,
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(text)
    print()

## Save GGUF (Optional - for llama.cpp / Ollama)

Uncomment and run if you want to create quantized GGUF files for local inference.

In [ ]:
# # Save as GGUF for local inference with llama.cpp or Ollama
# model.save_pretrained_gguf(
#     "dictalm2-israeli-law-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",  # Good balance of quality/size
# )

# # Push GGUF to HuggingFace
# model.push_to_hub_gguf(
#     f"{OUTPUT_NAME}-GGUF",
#     tokenizer,
#     quantization_method="q4_k_m",
#     token=True,
# )